[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/19_cost_and_caching.ipynb)

# 19. Cost and Caching

**Part 4: Reliability** · about 25 minutes

An agent that works but costs too much to run doesn't get used. This notebook is about the bill: how to measure it, how to cap it, and the single biggest lever for lowering it, **prompt caching**.

Caching rests on a fact you already know from notebook 04: every step resends the whole history. Step 10 sends everything step 9 sent, plus a little more. Most providers can recognise a beginning they've just seen and reuse the work they did on it. Those **cached** input tokens are usually billed at a large discount and processed faster. How big the discount is and how caching gets switched on varies by provider, so check yours.

There's one condition: the beginning has to be **exactly** the same, character for character. Change one character early in the request and everything after it is new again. So caching is less a feature you turn on than a discipline you follow, and this notebook measures how easy it is to break.

**By the end you will have:**

- a cost calculator and a per-run budget that stops runaway agents
- a tool that measures how much of each request a cache could reuse
- watched three innocent-looking habits destroy that number

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json
import os
from datetime import datetime

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. Pricing

Prices change often and differ by model and provider, so the numbers below are **examples only**. Put in the real prices for the model you use. The structure is what matters, and it's nearly universal:

- input tokens cost one price
- **cached** input tokens cost much less
- output tokens cost several times more than input

In [3]:
PRICES = {  # dollars per million tokens. EXAMPLE NUMBERS ONLY: replace them with your model's real prices.
    "input": 1.00,
    "cached_input": 0.10,
    "output": 5.00,
}


def cost(input_tokens, output_tokens, cached_tokens=0):
    fresh = input_tokens - cached_tokens
    return (fresh * PRICES["input"] + cached_tokens * PRICES["cached_input"]
            + output_tokens * PRICES["output"]) / 1_000_000

## 2. A budget that stops runaway agents

Notebook 04 capped the number of steps. A money cap is better, because steps vary wildly in size. One step that reads a huge file can cost more than twenty small ones. The agent checks its spend before every call and stops cleanly once the budget is used up.

We use the research agent from notebook 04, and record every request it sends, which section 3 needs.

In [4]:
NOTES = {
    "standup.md": "Standup notes. The launch deadline moved to March 14 because of the audit.",
    "owners.md": "Launch owner: Priya. Backend owner: Sam.",
    "lunch.md": "The team voted for dosa on Friday.",
    "handbook.md": "Team handbook.\n" + "\n".join(f"Rule {i}: be kind, write tests, keep pull requests small." for i in range(60)),
}
TOOLS = [
    {"type": "function", "function": {"name": "list_files", "description": "List the names of all note files.",
                                      "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {"name": "read_file", "description": "Read one note file by its exact name.",
                                      "parameters": {"type": "object", "properties": {"path": {"type": "string"}},
                                                     "required": ["path"]}}},
]
QUESTION = "What is the launch deadline, and who owns the launch?"


class BudgetExceeded(Exception):
    pass


def run_agent(system_prompt=lambda: "Answer using the tools. Read the files you need, then reply briefly.",
              before_call=None, tools=TOOLS, budget=None, max_steps=10):
    messages = [{"role": "system", "content": system_prompt()}, {"role": "user", "content": QUESTION}]
    spent, requests_sent, usages = 0.0, [], []
    for step in range(1, max_steps + 1):
        messages[0] = {"role": "system", "content": system_prompt()}  # rebuilt each step, as many agents do
        if before_call:
            messages = before_call(messages)
        if budget is not None and spent >= budget:
            raise BudgetExceeded(f"stopped before step {step}: spent ${spent:.6f} of a ${budget:.6f} budget")
        payload = {"model": llm.model, "messages": messages, "tools": tools}
        requests_sent.append(json.loads(json.dumps(payload)))  # a copy of exactly what was sent
        response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
        response.raise_for_status()
        data = response.json()
        usage = data.get("usage") or {}
        cached = (usage.get("prompt_tokens_details") or {}).get("cached_tokens", 0)
        usages.append(usage)
        spent += cost(usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0), cached)
        message = data["choices"][0]["message"]
        messages.append(message)
        if not message.get("tool_calls"):
            return {"answer": message.get("content"), "spent": spent, "requests": requests_sent, "usages": usages}
        for call in message["tool_calls"]:
            path = json.loads(call["function"]["arguments"] or "{}").get("path")
            result = json.dumps(sorted(NOTES)) if call["function"]["name"] == "list_files" \
                else NOTES.get(path, f"Error: no file named {path!r}")
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    return {"answer": None, "spent": spent, "requests": requests_sent, "usages": usages}


normal = run_agent()
print("Answer:", normal["answer"])
print(f"Cost of the run: ${normal['spent']:.6f} with the example prices"
      + (" (offline token counts are estimates)" if llm.offline else ""))

Answer: The launch deadline is March 14, and the owner is Priya.
Cost of the run: $0.003609 with the example prices (offline token counts are estimates)


In [5]:
tight = normal["spent"] / 3
try:
    run_agent(budget=tight)
except BudgetExceeded as error:
    print("Budget guard:", error)

Budget guard: stopped before step 4: spent $0.002209 of a $0.001203 budget


Notice that it stopped *after* going over: the last call it made took it past the limit. Checking before each call can't know what that call will cost. If you need a hard ceiling, estimate the next call's cost first, from the size of the history and the largest output you allow, and stop if it wouldn't fit. That's exercise 4.

Stopping is only half the job. Like the outage in notebook 16, the agent's messages could be checkpointed so a human can raise the budget and resume. What you must never do is let a run spend without limit and find out from the invoice.

## 3. How much of each request could be cached?

For each request, we measure how much of its beginning is **identical** to the request just before it. That's the part a cache could reuse. We compare the requests in the order most providers process them: tool definitions first, then the messages.

The columns are estimated tokens: everything **sent** over the whole run, the **fresh** part that has to be paid for in full, the share that could be **cached**, and the estimated input cost using the example prices. The very first request can never be cached, because there's nothing before it.

In [6]:
def as_prefix(payload):
    return json.dumps(payload["tools"], sort_keys=True) + json.dumps(payload["messages"], sort_keys=True)


def common_prefix(a, b):
    return len(os.path.commonprefix([a, b]))


def cacheable_report(requests_sent, label):
    texts = [as_prefix(p) for p in requests_sent]
    reused = sum(common_prefix(texts[i - 1], texts[i]) for i in range(1, len(texts)))
    sent = sum(len(t) for t in texts)
    fresh = sent - reused
    share = reused / sent
    estimate = (fresh // 4 * PRICES["input"] + reused // 4 * PRICES["cached_input"]) / 1_000_000
    print(f"{label:<34}{sent // 4:>9,}{fresh // 4:>9,}{100 * share:>9.1f}%{estimate:>13.6f}")
    return share, fresh


print(f"{'':<34}{'sent':>9}{'fresh':>9}{'cached':>10}{'input $':>13}")
baseline_share, baseline_fresh = cacheable_report(normal["requests"], "Normal run")

                                       sent    fresh    cached      input $
Normal run                            2,806    1,319     53.0%     0.001468


An agent loop only ever *appends* to the history, so each request starts with the whole previous one, and only the new part is fresh. Here the share isn't higher because this run is short and one step appended a big file, which is fresh the first time it's sent. On longer runs the share climbs towards 100%, since each new step adds little compared with everything before it.

Now three common habits that each look harmless.

### Habit 1: a timestamp in the system prompt

"Current time: ..." in the system prompt is popular, because models like knowing the date. But the system prompt comes near the very beginning, so every request now differs from the last one there.

In [7]:
with_clock = run_agent(system_prompt=lambda: "Answer using the tools. Read the files you need, then reply briefly. "
                                             f"Current time: {datetime.now().isoformat()}")
clock_share, clock_fresh = cacheable_report(with_clock["requests"], "Timestamp in the system prompt")

Timestamp in the system prompt        2,851    2,482     12.9%     0.002519


Almost everything after the tool definitions is now new on every call. If the model needs the date, give it with day precision, which changes once a day instead of every call. Or put it at the **end**, in the newest message, where it only affects what comes after it.

### Habit 2: editing the history

Clearing old tool results (notebook 12) saves tokens. Doing it on *every* step means the history changes somewhere in the middle each time, and everything from that point is new.

In [8]:
def clear_every_step(messages):
    tool_positions = [i for i, m in enumerate(messages) if m["role"] == "tool"]
    return [{**m, "content": "[cleared]"} if i in tool_positions[:-1] else m for i, m in enumerate(messages)]


with_clearing = run_agent(before_call=clear_every_step)
clearing_share, clearing_fresh = cacheable_report(with_clearing["requests"], "Clearing old results every step")

Clearing old results every step       2,812    1,498     46.7%     0.001629


Compare it with the normal run. Clearing was supposed to save tokens, yet here it sent about as many in total, because the cleared file list made the agent take an extra step to look again. And less of what it sent could come from the cache, because the history changed in the middle on every step. Whether that's a good trade depends on how big your provider's cache discount is, and the cost column shows it for the example prices. That's why notebook 12 said to compact *rarely and in large steps*: one big edit breaks the cache once, while a small edit on every step breaks it every step.

### Habit 3: tools in a different order

Some code builds the tool list from a `set` or a dictionary that isn't guaranteed to keep its order, so the list can come out in a different order from call to call. The tool definitions come **first**, so the whole request changes from the very first character.

In [9]:
# Take the normal run's requests and flip the tool order on every other one. Nothing else changes.
requests_with_flipping = []
for i, payload in enumerate(normal["requests"]):
    flipped = dict(payload, tools=TOOLS if i % 2 == 0 else list(reversed(TOOLS)))
    requests_with_flipping.append(flipped)
order_share, order_fresh = cacheable_report(requests_with_flipping, "Tool order changing between calls")

Tool order changing between calls     2,806    2,782      0.8%     0.002784


In [10]:
# Checkpoint: each habit must lower the share of input a cache could reuse.
assert clock_share < baseline_share and clock_fresh > baseline_fresh
assert clearing_share < baseline_share
assert order_share < 0.1 and order_fresh > baseline_fresh
print("Checkpoint passed.")

Checkpoint passed.


The fix is the same in all three cases: **stable things first, changing things last, and never edit the past more than you have to.** Build the tool list in a fixed order, keep the system prompt identical between calls, and only append to the history except when you deliberately compact.

## 4. What your provider actually reports

Our measurement says what *could* be cached. Whether it *was* depends on the provider. Many OpenAI-compatible APIs report it in `usage.prompt_tokens_details.cached_tokens`. Some cache automatically above a minimum length, others need you to mark where the cacheable part ends, and some local servers don't report it at all.

In [11]:
reported = [(u.get("prompt_tokens_details") or {}).get("cached_tokens") for u in normal["usages"]]
if any(reported):
    for step, (usage, cached) in enumerate(zip(normal["usages"], reported), 1):
        print(f"step {step}: {usage.get('prompt_tokens', 0):>6} input tokens, {cached or 0:>6} served from cache")
else:
    print("This provider didn't report cached tokens for these requests.")
    print("That can mean no caching, caching below a minimum length, or caching it doesn't report. Check its docs.")

This provider didn't report cached tokens for these requests.
That can mean no caching, caching below a minimum length, or caching it doesn't report. Check its docs.


## 5. The other levers

Caching makes repeated input cheap. Here's everything else that moves the bill, roughly in order of effect:

| Lever | Why it works | Where in this course |
|---|---|---|
| Fewer steps | Every step resends the history | Better prompts and tools (Parts 1 and 2) |
| Smaller tool results | Big results are paid for on every later step | Notebook 11 |
| Precise edits instead of rewrites | Output tokens cost the most | Notebook 06 |
| Compacting rarely and in big steps | Protects the cache | Notebook 12, and this notebook |
| A smaller model for simple calls | Summaries and classification rarely need the biggest model | Exercise 3 below |
| Shorter tool descriptions | They're sent on every call | Notebook 11 |

And measure **cost per solved task**, not cost per call. A cheaper model that needs twice the steps and solves fewer tasks is usually more expensive, and the eval suite from notebook 18 is how you find out.

## Recap

- Measure cost per run and per solved task, and give every run a budget.
- Prompt caching can make repeated input much cheaper, but only for an identical beginning.
- Keep stable content first: tools in a fixed order, then a system prompt that doesn't change.
- Timestamps, frequent history edits and shuffled tools quietly destroy caching.
- Check what your provider actually reports. Measured beats assumed.

## Exercises

1. Put the timestamp in the *last* user message instead of the system prompt. How much of the cacheable share comes back?
2. Use the eval suite from notebook 18 to compute cost per solved task for two different models, if you have access to two.
3. Route the compaction summary from notebook 12 to a smaller, cheaper model. How would you check that the summaries are still good enough?
4. Make the budget a hard ceiling: before each call, estimate its maximum cost and stop if it wouldn't fit. How much budget does that leave unused?

**Next:** [20. Agent v4](20_agent_v4.ipynb). Retries, tracing, budgets and evals in one agent, tested on a network that keeps failing.